In [1]:
from model_wrapper import *
from sklearn.externals.array_api_extra.testing import override

# of Training Instances: 47
# of Testing Instances: 11
Current RAM usage: 298.36 MB


In [2]:
import tensorflow as tf
import numpy as np

def load_data(folder_path, label):
    def process_instance(path_tensor):
        actual_path = path_tensor.numpy().decode('utf-8')
        x = get_training_instance(actual_path)
        x = np.reshape(x, (IMG_HEIGHT, MIN_IMG_COUNT * IMG_WIDTH, 1))
        return x.astype(np.float32)

    image_data = tf.py_function(
        func=process_instance,
        inp=[folder_path],
        Tout=tf.float32
    )

    image_data.set_shape([IMG_HEIGHT, MIN_IMG_COUNT * IMG_WIDTH, 1])

    return image_data, label

class CNNModel(Model):
    def __init__(self, anatomical_plane, fluid_sensitive=None, fat_suppression=None):
        self.model = tf.keras.models.Sequential([
            tf.keras.Input(shape=(IMG_HEIGHT, MIN_IMG_COUNT * IMG_WIDTH, 1)),
            tf.keras.layers.Conv2D(16, kernel_size=(5, 5), strides=(2, 4), activation="relu"),
            tf.keras.layers.MaxPool2D(pool_size=2),

            tf.keras.layers.Conv2D(32, 3, activation="relu"),
            tf.keras.layers.MaxPool2D(pool_size=2),

            tf.keras.layers.Conv2D(64, 3, activation="relu"),
            tf.keras.layers.MaxPool2D(pool_size=2),

            # 3. Generalize and save memory
            tf.keras.layers.GlobalAveragePooling2D(),
            tf.keras.layers.Dense(100, activation='relu'),
            tf.keras.layers.Dense(len(target_columns), activation="sigmoid")
        ])
        self.model.compile(loss="binary_crossentropy",
                optimizer=tf.keras.optimizers.Adam(),
                metrics=[tf.keras.metrics.AUC(multi_label=True, num_labels=len(target_columns), name='auc')])
        self.history = None
        super().__init__(anatomical_plane, fluid_sensitive, fat_suppression, full_train=False)

    @override
    def batch_fit(self, training_folders: pd.Series, y_train: np.ndarray, validation_folders: pd.Series, y_validation: np.ndarray):
        train_dataset = tf.data.Dataset.from_tensor_slices((training_folders, y_train))
        val_dataset = tf.data.Dataset.from_tensor_slices((validation_folders, y_validation))

        train_dataset = train_dataset.map(load_data, num_parallel_calls=tf.data.AUTOTUNE)
        train_dataset = train_dataset.batch(BATCH_SIZE)
        train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)

        val_dataset = val_dataset.map(load_data, num_parallel_calls=tf.data.AUTOTUNE)
        val_dataset = val_dataset.batch(BATCH_SIZE)
        val_dataset = val_dataset.prefetch(tf.data.AUTOTUNE)

        self.history = self.model.fit(
            train_dataset,
            epochs=15,
            validation_data=val_dataset
        )

    @override
    def predict_batch(self, x: np.ndarray) -> np.ndarray:
        x = np.reshape(x, shape=(x.shape[0], IMG_HEIGHT, MIN_IMG_COUNT * IMG_WIDTH, 1))
        return self.model.predict(x)

    @override
    def predict_instance(self, x: np.ndarray) -> np.ndarray:
        x = np.reshape(x, shape=(1, IMG_HEIGHT, MIN_IMG_COUNT * IMG_WIDTH, 1))
        pred_ = self.model.predict(x)
        return np.reshape(pred_, shape=(pred_.shape[1]))

I0000 00:00:1789173865.079606   25183 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
ensemble: List[Model] = []

for p in planes:
    for i in range(2):
        ensemble.append(CNNModel(p, i, i))

I0000 00:00:1789173873.262757   25183 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9709 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060, pci bus id: 0000:01:00.0, compute capability: 8.6


Training Model: 
	Plane: Sagittal
	Fluid Sensitive: 0
	Fat Suppression: 0
	Training Size: (36,)
	Validation Size: (20,)
Epoch 1/15


/home/zero/venvs/ds-venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1789173888.118630   25301 service.cc:153] XLA service 0x7dbaf80325e0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1789173888.118670   25301 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 3060, Compute Capability 8.6 (Driver: 13.3.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.24.0)
I0000 00:00:1789173888.157071   25301 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1789173888.496615   25301 cuda_dnn.cc:461] Loaded cuDNN version 92400
I0000 00:00:1789173888.518102   25301 dot_merger.cc:481] Merging Dots

8/8 ━━━━━━━━━━━━━━━━━━━━ 37s 2s/step - auc: 0.3838 - loss: 0.6926 - val_auc: 0.5596 - val_loss: 0.6801
Epoch 2/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 13s 1s/step - auc: 0.4274 - loss: 0.6793 - val_auc: 0.5890 - val_loss: 0.6568
Epoch 3/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 12s 1s/step - auc: 0.4146 - loss: 0.6588 - val_auc: 0.5894 - val_loss: 0.6195
Epoch 4/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 12s 1s/step - auc: 0.4298 - loss: 0.6384 - val_auc: 0.5870 - val_loss: 0.6059
Epoch 5/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 13s 2s/step - auc: 0.4367 - loss: 0.6272 - val_auc: 0.5792 - val_loss: 0.6080
Epoch 6/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 12s 1s/step - auc: 0.4260 - loss: 0.6248 - val_auc: 0.5809 - val_loss: 0.6090
Epoch 7/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 12s 1s/step - auc: 0.4162 - loss: 0.6230 - val_auc: 0.5726 - val_loss: 0.6093
Epoch 8/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 13s 1s/step - auc: 0.4140 - loss: 0.6218 - val_auc: 0.5855 - val_loss: 0.6111
Epoch 9/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 12s 1s/step - auc: 0.4071 - loss: 0.6208 - val_auc: 0.5840 - va

/home/zero/venvs/ds-venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1789174099.143313   25300 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_10511__.42


...

6/6 ━━━━━━━━━━━━━━━━━━━━ 33s 3s/step - auc: 0.4326 - loss: 0.6912 - val_auc: 0.5498 - val_loss: 0.6873
Epoch 2/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 11s 2s/step - auc: 0.4759 - loss: 0.6801 - val_auc: 0.5225 - val_loss: 0.6724
Epoch 3/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - auc: 0.4551 - loss: 0.6535 - val_auc: 0.5126 - val_loss: 0.6470
Epoch 4/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - auc: 0.4247 - loss: 0.6264 - val_auc: 0.5090 - val_loss: 0.6445
Epoch 5/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 11s 2s/step - auc: 0.4009 - loss: 0.6198 - val_auc: 0.5182 - val_loss: 0.6352
Epoch 6/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - auc: 0.4071 - loss: 0.6121 - val_auc: 0.5216 - val_loss: 0.6294
Epoch 7/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - auc: 0.4124 - loss: 0.6111 - val_auc: 0.5181 - val_loss: 0.6287
Epoch 8/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 11s 2s/step - auc: 0.4088 - loss: 0.6104 - val_auc: 0.5122 - val_loss: 0.6298
Epoch 9/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - auc: 0.4142 - loss: 0.6095 - val_auc: 0.5187 - va

/home/zero/venvs/ds-venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1789174523.776170   25300 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_31685__.42


6/6 ━━━━━━━━━━━━━━━━━━━━ 27s 3s/step - auc: 0.3659 - loss: 0.6897 - val_auc: 0.4741 - val_loss: 0.6741
Epoch 2/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 8s 1s/step - auc: 0.4281 - loss: 0.6681 - val_auc: 0.4819 - val_loss: 0.6364
Epoch 3/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 8s 1s/step - auc: 0.4288 - loss: 0.6371 - val_auc: 0.4877 - val_loss: 0.6108
Epoch 4/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 8s 1s/step - auc: 0.4167 - loss: 0.6370 - val_auc: 0.4842 - val_loss: 0.6201
Epoch 5/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 9s 1s/step - auc: 0.4559 - loss: 0.6334 - val_auc: 0.4796 - val_loss: 0.6128
Epoch 6/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 8s 1s/step - auc: 0.4752 - loss: 0.6229 - val_auc: 0.4826 - val_loss: 0.6079
Epoch 7/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 9s 1s/step - auc: 0.4647 - loss: 0.6197 - val_auc: 0.4908 - val_loss: 0.6066
Epoch 8/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 9s 1s/step - auc: 0.4598 - loss: 0.6198 - val_auc: 0.4949 - val_loss: 0.6068
Epoch 9/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 9s 1s/step - auc: 0.4522 - loss: 0.6210 - val_auc: 0.4977 - val_loss: 

/home/zero/venvs/ds-venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1789174676.367759   25302 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_39148__.42


7/7 ━━━━━━━━━━━━━━━━━━━━ 30s 3s/step - auc: 0.4608 - loss: 0.6855 - val_auc: 0.5188 - val_loss: 0.6726
Epoch 2/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - auc: 0.4815 - loss: 0.6568 - val_auc: 0.4821 - val_loss: 0.6381
Epoch 3/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 9s 1s/step - auc: 0.4466 - loss: 0.6227 - val_auc: 0.4650 - val_loss: 0.6175
Epoch 4/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 9s 1s/step - auc: 0.4150 - loss: 0.6144 - val_auc: 0.4814 - val_loss: 0.6138
Epoch 5/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - auc: 0.4270 - loss: 0.6115 - val_auc: 0.4853 - val_loss: 0.6147
Epoch 6/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - auc: 0.4367 - loss: 0.6125 - val_auc: 0.4941 - val_loss: 0.6149
Epoch 7/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - auc: 0.4321 - loss: 0.6127 - val_auc: 0.4926 - val_loss: 0.6136
Epoch 8/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - auc: 0.4374 - loss: 0.6124 - val_auc: 0.4823 - val_loss: 0.6126
Epoch 9/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - auc: 0.4308 - loss: 0.6119 - val_auc: 0.4942 - val_

In [4]:
scores = Model.get_ensemble_auc_score(ensemble, 1)

print(f"Model Score: {np.mean(scores)}")
for i in range(len(target_columns)):
    print(f"\t{target_columns[i]}: {scores[i]}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 505ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 507ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 534ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 503ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 488ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━

In [5]:
scores = Model.get_ensemble_auc_score(ensemble, 2)

print(f"Model Score: {np.mean(scores)}")
for i in range(len(target_columns)):
    print(f"\t{target_columns[i]}: {scores[i]}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━

In [6]:
data = {"name": []}
for t in target_columns:
    data[t] = []

for m in ensemble:
    data["name"].append(f"{m.anatomical_plane}_{m.fat_suppression}_{m.fluid_sensitive}")
    for i in range(len(target_columns)):
        data[target_columns[i]].append(m.auc_scores[i])

data = pd.DataFrame(data)
data

,name,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,Sagittal_0_0,0.642857,0.411765,0.291667,0.593750,0.666667,0.468750,0.642857,0.250000,0.505051,0.843750,0.654762,0.714286
1,Sagittal_1_1,0.649351,0.781250,0.389610,0.545455,0.323077,0.266667,0.388889,0.837500,0.064935,0.160714,0.353846,0.569444
2,Axial_0_0,0.500000,0.833333,1.000000,0.400000,0.500000,0.300000,0.500000,0.416667,0.600000,0.000000,0.416667,0.166667
3,Axial_1_1,0.542857,0.452381,0.625000,0.500000,0.550000,0.238095,0.514286,0.614286,0.375000,0.403846,0.769231,0.442857
4,Coronal_0_0,0.545455,0.366667,0.714286,0.514286,0.378788,0.307692,0.571429,0.257576,0.416667,0.309524,0.673077,0.884615
5,Coronal_1_1,0.584416,0.750000,0.862500,0.545455,0.400000,0.266667,0.402597,0.662338,0.580247,0.410714,0.476923,0.467532


In [7]:
auc_scores = []
for t in target_columns:
    pos = data[t].argmax()
    auc_scores.append(data[t].iloc[pos])
    print(f"{t}: score -> {data[t].iloc[pos]:.3f}, model -> {data['name'].iloc[pos]}")
print(f"Average: {np.mean(auc_scores)}")

ACL: score -> 0.649, model -> Sagittal_1_1
MCL: score -> 0.833, model -> Axial_0_0
Medial Meniscus: score -> 1.000, model -> Axial_0_0
Lateral Meniscus: score -> 0.594, model -> Sagittal_0_0
Medial OA: score -> 0.667, model -> Sagittal_0_0
Lateral OA: score -> 0.469, model -> Sagittal_0_0
PF OA: score -> 0.643, model -> Sagittal_0_0
Effusion: score -> 0.838, model -> Sagittal_1_1
Synovitis: score -> 0.600, model -> Axial_0_0
Baker's: score -> 0.844, model -> Sagittal_0_0
Contusion: score -> 0.769, model -> Axial_1_1
Fracture: score -> 0.885, model -> Coronal_0_0
Average: 0.7324836621711621
